**What is OPTIMIZE?**<br>
At its core, OPTIMIZE is a compaction mechanism. It takes many small files and merges them into larger, more efficient files.
**SQL Syntax:**<b>
OPTIMIZE table_name;

**Partial Optimization**: For very large tables, you don't always want to optimize the entire dataset. You can optimize specific partitions:<br>
OPTIMIZE table_name WHERE date >= '2024-01-01';

**Files Impact and Actions after Optimize:**
1. **Compaction and Re-writing**<br>
The OPTIMIZE process reads the data from many small files and writes that same data into new, larger files (compacted files). At this point, for a brief moment, you actually have double the data sitting in your storage: the original small files and the new large ones.

2. **"Tombstoning" in the Delta Log**<br>
Instead of deleting the physical files, Databricks updates the Delta Log (the _delta_log folder).<br>

- It adds a record saying: "These new large files are now part of the table."
- It adds a "Remove" action for the old small files.<br>
In Delta Lake terms, those old files are now tombstoned. They are marked as "removed" in the current version of the table, so any new queries will ignore them and only read the new, optimized files.

3. **Why are they kept?** (Time Travel)<br>
The reason the old files aren't deleted immediately is to support Point-in-Time Recovery (Time Travel).

If you realized you made a mistake and wanted to see the table exactly as it looked before the optimization, Databricks needs those old small files to reconstruct that previous version. If OPTIMIZE deleted them instantly, your version history would break.

4. **Physical Deletion via VACUUM**
The old files will stay in your storage indefinitely until you (or a background process) run the VACUUM command.

The Default: By default, VACUUM will only delete files that were tombstoned more than 7 days ago. This "Retention Period" ensures that no active, long-running queries are still trying to read those old files while you are trying to delete them.
command: VACUUM table_name RETAIN 168 HOURS; -- 168 hours is the 7-day default



**Summary of the Lifecycle**<br>
Before Optimize: You have 1,000 small files (
1
1 MB each).<br>
During Optimize: Databricks reads the 1,000 files and writes 1 large file (
1
1 GB).<br>
After Optimize: You have 1,001 files in storage. Your queries only see the 1 large file.<br>
After Vacuum: The 1,000 small files are permanently deleted from S3/ADLS, leaving only the 1 large file.

**Things to Consider While Optimize**:
-  Resource and Compute Intensity
-  Z-Order Column Limits (Rule upto 4 columns)
-  Storage "Double-Counting"